In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
data_path =  "../data/original/Bonn/reported_cases"
files = ["GeoHealth_20230314_Einzug_KA_Salierweg.xlsx", "GeoHealth_20240313_Einzug_KA_Salierweg.xlsx"]

In [ ]:
df = pd.read_excel(data_path + "/" + files[0], sheet_name=1)
aggregated_data = df[["Meldedatum", "Einzug_KA_Salierweg", "N"]].groupby(["Meldedatum", "Einzug_KA_Salierweg"]).sum().reset_index()

In [ ]:
df = pd.read_excel(data_path + "/" + files[1], sheet_name=0)
df.Meldedatum = df.Meldedatum.dt.floor("D").astype("datetime64[ns]")
aggregated_data_2 = df[["Meldedatum", "Einzug_KA_Salierweg", "N"]].groupby(["Meldedatum", "Einzug_KA_Salierweg"]).sum().reset_index()

In [ ]:
aggregated_data = pd.concat([aggregated_data, aggregated_data_2])

aggregated_data['Einzug_KA_Salierweg'] = (
    aggregated_data['Einzug_KA_Salierweg'].str.replace(' ', '', regex=False)
)

df = aggregated_data.copy()

# compute the 7-day sum inside each group.
seven_day = (
    df.set_index('Meldedatum')                       # date becomes the index
      .sort_index()
      .groupby('Einzug_KA_Salierweg')['N']           
      .rolling('7D', closed='both')                  # 7-day *calendar* window
      .sum()                                        
      .reset_index()                                 
      .rename(columns={'N': 'N_7d'})                 
)

df = df.merge(seven_day, on=['Meldedatum', 'Einzug_KA_Salierweg'])

# ensure that we only consider data for which we have a full 7-day window
df = df.loc[df.Meldedatum >= (aggregated_data.Meldedatum.min() + pd.Timedelta(days=6))]

df.to_csv("../data/preprocessed/" + "case_counts.csv", index=False)